# **EOG_Signal_Measurement_Implementation_Unit(I)**

> In this experiement, we will use the slope formula to analysis EOG signal.

> And use animation tools to turn the analysis result into animation to deepen our impression on EOG signal capability!

# 0. First import some formulas and functions we will be using!

In [ ]:
#@title
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rc
from matplotlib import animation

#eyemove_range = 1
#time_speed = 1
#fig = plt.figure(figsize=(12,5))
def animation_parameter(eyemove_range, time_speed):
  eyemove_range = eyemove_range/60
  time_speed = 25/time_speed
  return eyemove_range, time_speed


def drawframe(n):
    if(n+1 > np.size(data4)):
      n = np.size(data4)-2
    x1 = np.arange(0,n+1)
    EOG.set_data(x1,data4[0:n+1])

    x = np.linspace(0, 2, 1000)
    y1 = np.sin(2 * np.pi * (x - 0.01 * n))
    y2 = 0.7*np.cos(2 * np.pi * (x - 0.01 * n))

    eye1.set_data(y1[0:500]+1.5,y2[0:500])
    eyeball1.set_data((data4[n]*eyemove_range)+1.5,0)

    eye2.set_data(y1[0:500]-1.5,y2[0:500])
    eyeball2.set_data((data4[n]*eyemove_range)-1.5,0)
    txt_title.set_text('Frame = {0:4d}'.format(n))

    return eye1,

def animation_produce(time_speed):

  # equivalent to rcParams['animation.html'] = 'html5'
  rc('animation', html='html5')


  # blit=True re-draws only the parts that have changed.
  show_animation = animation.FuncAnimation(fig, drawframe, frames=500, interval=time_speed, blit=True)

  return show_animation
def create_screen(y_length):
  # create a figure and axes
  fig = plt.figure(figsize=(12,5))
  ax1 = plt.subplot(1,2,1)
  ax2 = plt.subplot(1,2,2)

  # set up the subplots as needed
  ax1.set_xlim(( 0, 500))
  ax1.set_ylim((-(y_length/2), (y_length/2)))
  #ax1.set_ylim((-200, 200))
  ax1.set_xlabel('Time')
  ax1.set_ylabel('Magnitude')

  ax2.set_xlim((-4,4))
  ax2.set_ylim((-2,2))
  ax2.set_xlabel('X')
  ax2.set_ylabel('Y')
  ax2.set_title('Eyeball Move')

  # create objects that will change in the animation. These are
  # initially empty, and will be given new values for each frame
  # in the animation.
  txt_title = ax1.set_title('')
  EOG, = ax1.plot([], [], 'b', lw=2)     # ax.plot returns a list of 2D line objects


  eyeball1, = ax2.plot([], [], 'k.', ms=70)
  eye1, = ax2.plot([], [], 'k', lw=4)
  eyeball2, = ax2.plot([], [], 'k.', ms=70)
  eye2, = ax2.plot([], [], 'k', lw=4)
  ax1.legend(['EOG']);
  return fig,ax1,ax2,txt_title,EOG,eyeball1,eye1,eyeball2,eye2


# 1. Enter your EOG signal of your left and right eyes into the codebase!

In [ ]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

In [ ]:
df = pd.read_csv('Sample_EOG_Signal.txt') #If using your own file, remember to change the file name in ('xxxx')!
df.info()
data = np.array(df)
data2 = data[0 : , 0]

# 2. Plot out the unprocessed EOG signal to see what it looks like!

In [ ]:
#Config the figure setting!
fig = go.Figure()
fig.add_trace(go.Scatter(
    y=data2, #Plot the EOG signal you measured
    mode='lines',
    name='Original Plot',
)
)
fig.update_layout(
    title="EOG signal you measured",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
fig.show()

# 3. Try putting the EOG into the slope formula to see the effect!

In [ ]:
def slopee(y1,y2):        #Set the slope function definition as a function

    x = (y2 - y1) / 2 #Middle sample frequency is 500 Hz. Data point interval is 2 ms
    return x


slopee_space = 30 #y2 and y1's data point interval (try to see the effect on the waveform through changing this value)

counter = slopee_space #Since we apply the slope formula with preceding data points. The starting point also needs to be shifted forward rather than starting at 0
data3 = np.zeros(np.size(data2)) #Creating space to store the calculated slope
while(counter < np.size(data2)):
  data3[counter] = slopee(data2[counter], data2[counter-slopee_space])
  counter += 1

In [ ]:
#Config the figure setting!
fig = go.Figure()
fig.add_trace(go.Scatter(
    y=data3, #Plot the EOG signal after applying the slope formula
    mode='lines',
    name='Original Plot',
)
)
fig.update_layout(
    title="EOG signal after applying the slope formula",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
fig.show()

> If y1, y2 interval is appropriate. The waveform above will signify eye movement (waveform moving up indicate eye looking right).


# 4. Try representing the EOG result in an animation!

> Since the sampling frequency for the EOG data is slightly high (500Hz). Normal animation don't need that much frames (25~50 fps is good enough).

> We can try to modify the data and only take one out of every ten points to get a new lower freqency data (50 fps). This is called downsampling.

In [ ]:
resampler = 10 #Downsampling multiple

data4 = np.zeros(int(np.size(data3)/resampler)) #Creating space for data after downsampling

counter = 0 #Used to track the position of the data after downsampling
counter_resample = 1 #A counter for when to resample

for i in data3:
  if(counter_resample == resampler): #Resample whenever the counter get to the value of resampler
    data4[counter] = i
    counter += 1
    counter_resample = 0
  counter_resample += 1

#Config the figure setting!
fig = go.Figure()
fig.add_trace(go.Scatter(
    y=data4, #Plot the EOG signal after downsampling!
    mode='lines',
    name='Original Plot',
)
)
fig.update_layout(
    title="EOG signal after downsampling",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
fig.show()

> Using the downsampled data, create an eyeball movement animation below!

In [ ]:
(fig,ax1,ax2,txt_title,EOG,eyeball1,eye1,eyeball2,eye2) = create_screen(200) #This 200 is used to set the canvas height (y-axis). If your data is narrower or wider you can adjust this value

(eyemove_range, time_speed) = animation_parameter(1, 1) #(Eyeball movement amplitude, animation speed) Each person's signal strength is different. Narrowly adjust the variable for the best animation effect!
show_animation = animation_produce(time_speed) #Setting animation setting

In [ ]:
show_animation #Start producing animation! Production and rendering should only take 1~2 minute!

# Unit Summary
---

1.   EOG signal analysis is not difficult. Understanding that our head have other electric signal other than brain waves.
2.   Downsampling can help applying the data effectively in different application.
3.   Animation is so cute.